# Open Notebook & Additional Resources

<a target="_blank" href="https://colab.research.google.com/github/Nicolepcx/ORM_AI_Agents_Bootcamp/blob/main/demo/DAY_1_DEMO_SESSION_5_communication_patterns.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>
<a target="_blank" href="https://learning.oreilly.com/library/view/ai-agents-the/0642572247775/">
  <img src="https://img.shields.io/badge/AI%20Agents%20Book-Read%20on%20O'Reilly-d40101?style=flat" alt="AI Agents Book – Read on O'Reilly"/>
</a>



# About the Notebook

This notebook focuses on communication design in multi-agent systems.

We use a real LLM through OpenAI, but keep the tasks generic so the protocol is the main thing to notice:
A) Message passing with explicit handoffs
B) Shared memory with centralized state
C) Event-driven communication with publish-subscribe routing

Each section shows:
- the state shape
- the Pydantic schema
- the handoff or routing logic
- the context-window implications

The point is not the domain. The point is how information moves.

In [ ]:
# Minimal dependencies for real LLM-backed protocol demos.
# If you install into a fresh runtime, rerun this cell once and then restart the kernel.
!pip install -q "langchain-openai" "python-dotenv>=1.0,<2.0" "pydantic>=2,<3"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 3.1 MB/s eta 0:00:00


In [ ]:
from __future__ import annotations

import json
import os
import uuid
from datetime import datetime, timezone
from typing import Any, Literal, Optional

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-nano")

llm: Optional[ChatOpenAI] = None
if OPENAI_API_KEY:
    llm = ChatOpenAI(model=OPENAI_MODEL, api_key=OPENAI_API_KEY, temperature=0)
    print(f"Using model: {OPENAI_MODEL}")
else:
    print("Set OPENAI_API_KEY to run the LLM-backed demos.")

Using model: gpt-5.4-nano


# Shared Helpers

In [ ]:
WORK_ITEMS = [
    "triage a new request",
    "review a proposed change",
    "update a shared document",
    "route a new event",
]


def sample_item(index: int) -> str:
    return WORK_ITEMS[index % len(WORK_ITEMS)]


def require_llm() -> ChatOpenAI:
    if llm is None:
        raise ValueError("OPENAI_API_KEY is not set. Add it to your environment or .env file first.")
    return llm


def new_id(prefix: str) -> str:
    return f"{prefix}_{uuid.uuid4().hex[:8]}"


def now_ts() -> str:
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def dump_json(value: Any) -> str:
    if isinstance(value, BaseModel):
        return value.model_dump_json(indent=2)
    return json.dumps(value, indent=2, default=str)


def payload_chars(value: Any) -> int:
    return len(dump_json(value))


class WorkBrief(BaseModel):
    work_id: str
    objective: str
    constraints: list[str]
    acceptance_criteria: list[str]
    context_budget_note: str

# Collaboration Patterns vs Communication Protocols

Collaboration patterns define who works with whom.
Communication protocols define how information flows.

A collaboration pattern sets the roles and relationships.
A communication protocol sets the handoffs, visibility, and synchronization rules.

You can keep the same collaboration pattern and swap the protocol underneath it.
For example, a planner and reviewer can coordinate by direct messages, a shared board, or events.

# Why This Matters

Poor communication design causes:
- Latency spikes
- State inconsistencies
- Error amplification
- Silent failure

This is where many production MAS collapse.

Context management is part of the problem:
- If every agent sees everything, the context window fills with irrelevant state.
- If agents see too little, handoffs become ambiguous and failures go silent.
- Good protocol design decides what must be visible, when, and to whom.

## 1) Message Passing

State shape:
- Explicit point-to-point communication
- Each message contains structured intent and metadata
- No shared global memory
- Everything is explicit

Strengths:
- Strong modularity and traceability
- Easy to audit handoffs and contracts

Trade-offs:
- Routing overhead
- Coordination cost at scale

Common examples:
- Tool invocation
- API orchestration
- Workflow pipelines
- Task delegation with explicit contracts

In [ ]:
class MessageMetadata(BaseModel):
    correlation_id: str
    message_id: str
    reply_to: Optional[str] = None
    priority: Literal["low", "normal", "high"] = "normal"
    requires_ack: bool = True
    timestamp: str


class MessageEnvelope(BaseModel):
    sender: str
    recipient: str
    intent: Literal["delegate", "review", "approve", "reject", "clarify"]
    metadata: MessageMetadata
    contract: WorkBrief
    payload: str


class ReviewDecision(BaseModel):
    approved: bool
    reasons: list[str] = Field(default_factory=list)
    revised_instruction: str
    minimal_context_needed: list[str] = Field(default_factory=list)


def demo_message_passing() -> dict[str, Any]:
    llm = require_llm()
    handoff_llm = llm.with_structured_output(MessageEnvelope)
    reviewer_llm = llm.with_structured_output(ReviewDecision)

    brief = WorkBrief(
        work_id=new_id("work"),
        objective="Prepare a safe execution plan for triaging a customer request.",
        constraints=[
            "Do not invent facts outside the brief.",
            "Keep the plan short enough for a downstream tool call.",
            "List at most three concrete steps.",
        ],
        acceptance_criteria=[
            "The reviewer can validate the plan from the handoff alone.",
            "The final instruction is executable without shared memory.",
        ],
        context_budget_note="Only pass the task contract and the current handoff. Do not attach unrelated history.",
    )

    handoff = handoff_llm.invoke(
        f"""You are a coordinator in a message-passing system.
Create a point-to-point handoff from planner to reviewer.
Everything must be explicit because there is no shared global memory.

Set:
- sender = planner
- recipient = reviewer
- intent = review
- priority = normal
- requires_ack = true

Task contract:
{dump_json(brief)}

Write a compact payload with the proposed action and what the reviewer should check.
"""
    )

    decision = reviewer_llm.invoke(
        f"""You are the reviewer.
Review the handoff below and decide whether it is safe to execute.
Call out the minimal context needed to make this decision.

Handoff:
{dump_json(handoff)}
"""
    )

    reply = MessageEnvelope(
        sender="reviewer",
        recipient="planner",
        intent="approve" if decision.approved else "reject",
        metadata=MessageMetadata(
            correlation_id=handoff.metadata.correlation_id,
            message_id=new_id("msg"),
            reply_to=handoff.metadata.message_id,
            priority=handoff.metadata.priority,
            requires_ack=False,
            timestamp=now_ts(),
        ),
        contract=brief,
        payload=decision.revised_instruction,
    )

    print("\n=== Message passing ===")
    print("Task contract:")
    print(dump_json(brief))
    print("\nPlanner -> reviewer handoff:")
    print(dump_json(handoff))
    print("\nReviewer decision:")
    print(dump_json(decision))
    print("\nReviewer -> planner reply:")
    print(dump_json(reply))
    print(f"\nContext sent to reviewer: {payload_chars({'brief': brief.model_dump(), 'handoff': handoff.model_dump()})} chars")

    return {
        "brief": brief,
        "handoff": handoff,
        "decision": decision,
        "reply": reply,
        "context_chars": payload_chars({"brief": brief.model_dump(), "handoff": handoff.model_dump()}),
    }

## 2) Shared Memory Approaches

State shape:
- Centralized state model
- Agents read and write to a shared memory object
- Everyone sees current state

Strengths:
- Transparent
- Easy to audit
- Good for knowledge aggregation and long-lived context

Trade-offs:
- Synchronization issues
- Central bottlenecks
- Context bloat if every agent receives the whole board every time

Common examples:
- Knowledge aggregation
- Blackboard systems
- Long-lived context storage
- System-wide state tracking

In [ ]:
class SharedSection(BaseModel):
    section_id: str
    title: str
    content: str
    revision: int
    last_updated_by: str


class SharedComment(BaseModel):
    author: str
    section_id: str
    comment: str
    based_on_revision: int


class SharedDocument(BaseModel):
    document_id: str
    title: str
    sections: list[SharedSection]
    comments: list[SharedComment] = Field(default_factory=list)
    global_notes: list[str] = Field(default_factory=list)
    revision: int


class SharedUpdate(BaseModel):
    agent: str
    seen_revision: int
    update_type: Literal["content", "formatting", "comment"]
    target_section_id: Optional[str] = None
    new_content: str
    rationale: str


def apply_shared_update(document: SharedDocument, update: SharedUpdate) -> tuple[SharedDocument, str]:
    next_doc = document.model_copy(deep=True)
    sync_note = "fresh read"
    if update.seen_revision != document.revision:
        sync_note = f"stale read: agent saw v{update.seen_revision}, board is already v{document.revision}"

    if update.update_type in {"content", "formatting"} and update.target_section_id:
        for section in next_doc.sections:
            if section.section_id == update.target_section_id:
                section.content = update.new_content
                section.revision += 1
                section.last_updated_by = update.agent
                break
    else:
        next_doc.comments.append(
            SharedComment(
                author=update.agent,
                section_id=update.target_section_id or "general",
                comment=update.new_content,
                based_on_revision=update.seen_revision,
            )
        )

    next_doc.revision += 1
    next_doc.global_notes.append(f"{update.agent}: {update.rationale} ({sync_note})")
    return next_doc, sync_note


def demo_shared_memory() -> dict[str, Any]:
    llm = require_llm()
    update_llm = llm.with_structured_output(SharedUpdate)

    document = SharedDocument(
        document_id=new_id("doc"),
        title="Collaborative operating note",
        sections=[
            SharedSection(
                section_id="section_a",
                title="Action plan",
                content="Draft the first response and keep it short.",
                revision=1,
                last_updated_by="system",
            )
        ],
        revision=1,
    )

    writer_snapshot = document.model_copy(deep=True)
    commenter_snapshot = document.model_copy(deep=True)

    writer_update = update_llm.invoke(
        f"""You are an agent editing a shared document.
Return a structured update.

Set:
- agent = writer
- seen_revision = {writer_snapshot.revision}
- update_type = content
- target_section_id = section_a

Shared document snapshot:
{dump_json(writer_snapshot)}

Rewrite the action plan so it includes a clear validation step.
"""
    )

    commenter_update = update_llm.invoke(
        f"""You are another agent reading the same shared document snapshot.
Return a structured update.

Set:
- agent = commenter
- seen_revision = {commenter_snapshot.revision}
- update_type = comment
- target_section_id = section_a

Shared document snapshot:
{dump_json(commenter_snapshot)}

Leave one short comment about what could still go wrong.
"""
    )

    document, writer_sync = apply_shared_update(document, writer_update)
    document, commenter_sync = apply_shared_update(document, commenter_update)

    print("\n=== Shared memory ===")
    print("Initial board snapshot sent to writer:")
    print(dump_json(writer_snapshot))
    print("\nWriter update:")
    print(dump_json(writer_update))
    print(f"Sync note after writer update: {writer_sync}")
    print("\nCommenter update:")
    print(dump_json(commenter_update))
    print(f"Sync note after commenter update: {commenter_sync}")
    print("\nFinal shared document:")
    print(dump_json(document))
    print(f"\nContext sent to each shared-memory agent: {payload_chars(writer_snapshot)} chars")

    return {
        "initial_document": writer_snapshot,
        "writer_update": writer_update,
        "commenter_update": commenter_update,
        "final_document": document,
        "context_chars": payload_chars(writer_snapshot),
        "sync_notes": [writer_sync, commenter_sync],
    }

## 3) Event-Driven Coordination

State shape:
- Producers emit events with structured payload plus metadata
- Subscribers react without direct point-to-point calls
- Roles stay decoupled

Strengths:
- Fan-out parallelism
- Decoupled roles
- Good audit trail through event logs
- Good fit when agents should not run continuously

Trade-offs:
- Harder end-to-end tracing unless you keep event IDs and delivery logs
- Silent failure if subscriptions or handlers are missing

Common examples:
- Asynchronous approval or escalation paths
- Event-log based audit trails
- Systems that should avoid polling and reduce idle compute

In [ ]:
class CoordinationEvent(BaseModel):
    event_id: str
    event_type: Literal["task_created", "constraint_changed", "approval_requested"]
    producer: str
    timestamp: str
    payload: dict[str, Any]
    metadata: dict[str, Any] = Field(default_factory=dict)


class Subscription(BaseModel):
    event_type: str
    handler: str
    purpose: str


class HandlerResult(BaseModel):
    handler: str
    should_act: bool
    action: str
    reason: str


class DeliveryRecord(BaseModel):
    event_id: str
    handler: str
    delivered: bool
    result_summary: str


SUBSCRIPTIONS = [
    Subscription(event_type="constraint_changed", handler="planner", purpose="adjust the current plan"),
    Subscription(event_type="constraint_changed", handler="auditor", purpose="record the risk and escalation trail"),
]


def demo_event_driven() -> dict[str, Any]:
    llm = require_llm()
    handler_llm = llm.with_structured_output(HandlerResult)

    event = CoordinationEvent(
        event_id=new_id("evt"),
        event_type="constraint_changed",
        producer="scheduler",
        timestamp=now_ts(),
        payload={
            "item": sample_item(3),
            "change": "Approval is now required before execution.",
        },
        metadata={"priority": "high", "correlation_id": new_id("corr")},
    )

    matching_subscriptions = [s for s in SUBSCRIPTIONS if s.event_type == event.event_type]
    delivery_log: list[DeliveryRecord] = []
    handler_outputs: list[HandlerResult] = []

    print("\n=== Event driven ===")
    print("Published event:")
    print(dump_json(event))
    print("\nMatching subscriptions:")
    print(dump_json([s.model_dump() for s in matching_subscriptions]))

    for subscription in matching_subscriptions:
        result = handler_llm.invoke(
            f"""You are an event subscriber.
Return a structured handler result.

Handler name: {subscription.handler}
Handler purpose: {subscription.purpose}

Event:
{dump_json(event)}

Decide whether this handler should act, and if so, what it should do.
Only use the event payload and metadata you received.
"""
        )
        handler_outputs.append(result)
        delivery_log.append(
            DeliveryRecord(
                event_id=event.event_id,
                handler=subscription.handler,
                delivered=True,
                result_summary=result.action,
            )
        )
        print(f"\nHandler result for {subscription.handler}:")
        print(dump_json(result))

    print("\nDelivery log:")
    print(dump_json([record.model_dump() for record in delivery_log]))
    print(f"\nContext sent to each subscriber: {payload_chars(event)} chars")

    return {
        "event": event,
        "subscriptions": matching_subscriptions,
        "handler_outputs": handler_outputs,
        "delivery_log": delivery_log,
        "context_chars": payload_chars(event),
    }

# Context Window Pressure

Communication design is also context design.

The first pass below is intentionally disciplined:
- Message passing uses a compact contract plus one handoff.
- Shared memory uses a tiny board.
- Event-driven uses one small event.

That is useful for showing the ideal form, but not the failure mode.
So after the clean run, we inject anti-patterns to force the breakdowns:
- message passing: copied history and too many hops
- shared memory: stale overwrites and bloated boards
- event-driven: dropped subscriptions and fan-out amplification

In [ ]:
def compare_context_pressure(*, message_demo: dict[str, Any], shared_demo: dict[str, Any], event_demo: dict[str, Any]) -> None:
    print("\n=== Context window comparison: clean design ===")
    print(f"message passing payload: {message_demo['context_chars']} chars")
    print(f"shared memory payload:   {shared_demo['context_chars']} chars")
    print(f"event-driven payload:    {event_demo['context_chars']} chars")

    print("\nInterpretation:")
    print("- These are disciplined examples, so all three payloads stay relatively small.")
    print("- Small payloads here do not mean the pattern is always cheap in production.")
    print("- The next cell intentionally forces bad protocol choices to surface the real failure modes.")


if llm is None:
    print("Set OPENAI_API_KEY, then rerun the notebook from the top to execute the demos.")
else:
    message_demo = demo_message_passing()
    shared_demo = demo_shared_memory()
    event_demo = demo_event_driven()
    compare_context_pressure(
        message_demo=message_demo,
        shared_demo=shared_demo,
        event_demo=event_demo,
    )


=== Message passing ===
Task contract:
{
  "work_id": "work_7cb829c5",
  "objective": "Prepare a safe execution plan for triaging a customer request.",
  "constraints": [
    "Do not invent facts outside the brief.",
    "Keep the plan short enough for a downstream tool call.",
    "List at most three concrete steps."
  ],
  "acceptance_criteria": [
    "The reviewer can validate the plan from the handoff alone.",
    "The final instruction is executable without shared memory."
  ],
  "context_budget_note": "Only pass the task contract and the current handoff. Do not attach unrelated history."
}

Planner -> reviewer handoff:
{
  "sender": "planner",
  "recipient": "reviewer",
  "intent": "review",
  "metadata": {
    "correlation_id": "corr_work_7cb829c5",
    "message_id": "msg_review_001",
    "reply_to": null,
    "priority": "normal",
    "requires_ack": true,
    "timestamp": "2026-03-20T00:00:00Z"
  },
  "contract": {
    "work_id": "work_7cb829c5",
    "objective": "Prepare a s

In [ ]:
def stress_message_passing(clean_demo: dict[str, Any]) -> dict[str, Any]:
    clean_handoff = clean_demo["handoff"].model_dump()
    forwarded_packet = {
        "current_handoff": clean_handoff,
        "routing_history": [
            {"hop": 1, "from": "planner", "to": "coordinator", "note": "initial handoff"},
            {"hop": 2, "from": "coordinator", "to": "validator", "note": "copied full prior packet for safety"},
            {"hop": 3, "from": "validator", "to": "reviewer", "note": "copied full prior packet again plus audit notes"},
        ],
        "duplicated_contracts": [clean_handoff["contract"] for _ in range(4)],
        "duplicated_history": [clean_handoff for _ in range(3)],
        "extra_notes": [
            "include all prior context just in case",
            "do not trim anything",
            "forward every previous payload verbatim",
        ] * 15,
    }
    bloated_chars = payload_chars(forwarded_packet)
    return {
        "clean_chars": clean_demo["context_chars"],
        "bloated_chars": bloated_chars,
        "hop_count": 3,
        "growth": bloated_chars - clean_demo["context_chars"],
    }


def stress_shared_memory(clean_demo: dict[str, Any]) -> dict[str, Any]:
    base_doc = clean_demo["final_document"].model_copy(deep=True)
    for idx in range(12):
        base_doc.sections.append(
            SharedSection(
                section_id=f"extra_{idx}",
                title=f"Irrelevant section {idx}",
                content=("background detail that most agents do not need right now. ") * 8,
                revision=1,
                last_updated_by="archiver",
            )
        )
    base_doc.global_notes.extend([f"historical note {i}" for i in range(80)])

    stale_snapshot_revision = base_doc.revision
    writer_update = SharedUpdate(
        agent="writer_a",
        seen_revision=stale_snapshot_revision,
        update_type="content",
        target_section_id="section_a",
        new_content="Action plan with approval gate and validation checklist.",
        rationale="Add the missing approval and validation requirements.",
    )
    formatter_update = SharedUpdate(
        agent="writer_b",
        seen_revision=stale_snapshot_revision,
        update_type="content",
        target_section_id="section_a",
        new_content="Short response only.",
        rationale="Simplify the section based on an outdated snapshot.",
    )

    after_writer, note_1 = apply_shared_update(base_doc, writer_update)
    after_formatter, note_2 = apply_shared_update(after_writer, formatter_update)
    final_section = next(section for section in after_formatter.sections if section.section_id == "section_a")

    return {
        "bloated_chars": payload_chars(base_doc),
        "clean_chars": clean_demo["context_chars"],
        "note_1": note_1,
        "note_2": note_2,
        "final_section": final_section.model_dump(),
        "lost_update": "approval gate" not in final_section.content.lower(),
    }


def stress_event_driven(clean_demo: dict[str, Any]) -> dict[str, Any]:
    clean_event = clean_demo["event"].model_dump()
    dropped_event = {
        **clean_event,
        "event_type": "constraint_change",  # typo: no subscriber listens to this
    }
    dropped_matches = [sub for sub in SUBSCRIPTIONS if sub.event_type == dropped_event["event_type"]]

    noisy_subscriptions = SUBSCRIPTIONS + [
        Subscription(event_type="constraint_changed", handler=f"shadow_handler_{i}", purpose="also react")
        for i in range(6)
    ]
    fanout_matches = [sub for sub in noisy_subscriptions if sub.event_type == clean_event["event_type"]]

    duplicate_actions = [
        f"{sub.handler} triggers follow-up work for correlation_id {clean_event['metadata']['correlation_id']}"
        for sub in fanout_matches
    ]

    return {
        "clean_chars": clean_demo["context_chars"],
        "event_chars": payload_chars(clean_event),
        "dropped_subscribers": len(dropped_matches),
        "fanout_subscribers": len(fanout_matches),
        "duplicate_actions": duplicate_actions[:4],
    }


if llm is not None:
    print("\n=== Failure injection: message passing ===")
    stressed_mp = stress_message_passing(message_demo)
    print(f"clean payload:   {stressed_mp['clean_chars']} chars")
    print(f"bloated payload: {stressed_mp['bloated_chars']} chars")
    print(f"extra payload from copied history: {stressed_mp['growth']} chars")
    print(f"handoff hops before decision: {stressed_mp['hop_count']}")
    print("Problem shown: routing overhead and copied history can make explicit handoffs expensive.")

    print("\n=== Failure injection: shared memory ===")
    stressed_sm = stress_shared_memory(shared_demo)
    print(f"clean board payload:   {stressed_sm['clean_chars']} chars")
    print(f"bloated board payload: {stressed_sm['bloated_chars']} chars")
    print(f"writer sync note: {stressed_sm['note_1']}")
    print(f"formatter sync note: {stressed_sm['note_2']}")
    print("final section after stale overwrite:")
    print(dump_json(stressed_sm['final_section']))
    print(f"lost update happened: {stressed_sm['lost_update']}")
    print("Problem shown: centralized state creates both context bloat and stale-write inconsistency.")

    print("\n=== Failure injection: event driven ===")
    stressed_ev = stress_event_driven(event_demo)
    print(f"clean event payload: {stressed_ev['clean_chars']} chars")
    print(f"subscribers after event-type typo: {stressed_ev['dropped_subscribers']}")
    print(f"subscribers after noisy fan-out: {stressed_ev['fanout_subscribers']}")
    print("sample duplicate actions:")
    print(dump_json(stressed_ev['duplicate_actions']))
    print("Problem shown: event-driven systems can fail silently or amplify work through uncontrolled fan-out.")


=== Failure injection: message passing ===
clean payload:   2024 chars
bloated payload: 10628 chars
extra payload from copied history: 8604 chars
handoff hops before decision: 3
Problem shown: routing overhead and copied history can make explicit handoffs expensive.

=== Failure injection: shared memory ===
clean board payload:   350 chars
bloated board payload: 10549 chars
writer sync note: fresh read
formatter sync note: stale read: agent saw v3, board is already v4
final section after stale overwrite:
{
  "section_id": "section_a",
  "title": "Action plan",
  "content": "Short response only.",
  "revision": 4,
  "last_updated_by": "writer_b"
}
lost update happened: True
Problem shown: centralized state creates both context bloat and stale-write inconsistency.

=== Failure injection: event driven ===
clean event payload: 336 chars
subscribers after event-type typo: 0
subscribers after noisy fan-out: 8
sample duplicate actions:
[
  "planner triggers follow-up work for correlation_id 